# (5) Attention Mechanisms

This chapter explores attention mechanisms and encoder-decoder architectures for enhancing RNN-based motor performance map prediction. We focus on how attention can improve the model's ability to focus on relevant operating conditions and capture complex dependencies in motor behavior.

## Learning Objectives

- Understand the limitations of fixed-size context vectors in traditional encoder-decoder models
- Master attention mechanism fundamentals and mathematical formulation
- Implement multi-head attention for motor sequence processing
- Learn encoder-decoder architectures for performance map prediction
- Explore cross-attention between operating conditions and design parameters

## 5.1 Motivation for Attention in Motor Performance Prediction

### 5.1.1 Limitations of Fixed-Size Context Vectors

Traditional encoder-decoder models compress the entire input sequence into a fixed-size context vector, which presents several challenges for motor performance prediction:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("📚 Attention Mechanisms for Motor Performance Map Prediction")
print("=" * 60)
print("🎯 Chapter Goals:")
print("  • Understand attention fundamentals")
print("  • Implement encoder-decoder with attention")
print("  • Apply multi-head attention to motor sequences")
print("  • Build cross-attention mechanisms")

## 5.2 Fundamentals of Attention Mechanisms

### 5.2.1 Mathematical Foundation

Attention mechanisms compute weighted sums of values, where weights are determined by compatibility between query and key vectors:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$: Query vectors (what we're looking for)
- $K$: Key vectors (what we have)
- $V$: Value vectors (what we return)
- $d_k$: Dimension of key vectors

In [ ]:
def attention_mechanism_demo():
    """Demonstrate basic attention mechanism with motor parameters"""
    
    # Sample motor operating conditions as sequences
    speeds = np.array([1000, 2000, 3000, 4000, 5000])  # RPM
    torques = np.array([50, 100, 150, 120, 80])      # Nm
    currents = np.array([30, 60, 90, 75, 50])       # A
    
    # Create input matrix (sequence_length x feature_dim)
    input_sequence = np.column_stack([speeds, torques, currents])
    sequence_length, feature_dim = input_sequence.shape
    
    # Random weight matrices for demonstration
    np.random.seed(42)
    d_k = feature_dim  # Key dimension
    
    # Query, Key, Value transformations
    W_q = np.random.randn(feature_dim, d_k) * 0.1
    W_k = np.random.randn(feature_dim, d_k) * 0.1
    W_v = np.random.randn(feature_dim, d_k) * 0.1
    
    # Compute Q, K, V
    Q = input_sequence @ W_q  # (seq_len, d_k)
    K = input_sequence @ W_k  # (seq_len, d_k)
    V = input_sequence @ W_v  # (seq_len, d_k)
    
    # Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)
    attention_weights = np.exp(scores) / np.sum(np.exp(scores), axis=1, keepdims=True)
    
    # Compute context vectors
    context_vectors = attention_weights @ V  # (seq_len, d_k)
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Attention Mechanism Demonstration', fontsize=16, fontweight='bold')
    
    # Plot 1: Input sequence
    ax1 = axes[0, 0]
    ax1.plot(speeds, torques, 'bo-', label='Torque', linewidth=2, markersize=8)
    ax1.set_xlabel('Speed (RPM)', fontweight='bold')
    ax1.set_ylabel('Torque (Nm)', fontweight='bold', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.set_title('Input Operating Conditions', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Twin axis for current
    ax1_twin = ax1.twinx()
    ax1_twin.plot(speeds, currents, 'ro--', label='Current', linewidth=2, markersize=8)
    ax1_twin.set_ylabel('Current (A)', fontweight='bold', color='r')
    ax1_twin.tick_params(axis='y', labelcolor='r')
    ax1_twin.legend(loc='upper right')
    
    # Plot 2: Attention weights heatmap
    ax2 = axes[0, 1]
    im = ax2.imshow(attention_weights, cmap='Blues', aspect='auto')
    ax2.set_xlabel('Key Position', fontweight='bold')
    ax2.set_ylabel('Query Position', fontweight='bold')
    ax2.set_title('Attention Weights Matrix', fontweight='bold')
    ax2.set_xticks(range(sequence_length))
    ax2.set_yticks(range(sequence_length))
    ax2.set_xticklabels([f'{s} RPM' for s in speeds], rotation=45)
    ax2.set_yticklabels([f'{s} RPM' for s in speeds])
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax2)
    cbar.set_label('Attention Weight', fontweight='bold')
    
    # Plot 3: Attention distribution for each query
    ax3 = axes[1, 0]
    for i in range(sequence_length):
        ax3.bar(range(sequence_length), attention_weights[i], 
               alpha=0.7, label=f'Query {i+1} ({speeds[i]} RPM)')
    
    ax3.set_xlabel('Key Position', fontweight='bold')
    ax3.set_ylabel('Attention Weight', fontweight='bold')
    ax3.set_title('Attention Distribution per Query', fontweight='bold')
    ax3.set_xticks(range(sequence_length))
    ax3.set_xticklabels([f'{s}' for s in speeds])
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Context vectors
    ax4 = axes[1, 1]
    for i in range(d_k):
        ax4.plot(speeds, context_vectors[:, i], 'o-', 
                label=f'Context Dim {i+1}', linewidth=2, markersize=6)
    
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Context Vector Value', fontweight='bold')
    ax4.set_title('Resulting Context Vectors', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print insights
    print("🧠 Attention Mechanism Insights:")
    print("=" * 40)
    print(f"📊 Input sequence length: {sequence_length}")
    print(f"🔢 Feature dimension: {feature_dim}")
    print(f"🎯 Key dimension (d_k): {d_k}")
    print(f"⚖️ Attention weights shape: {attention_weights.shape}")
    print(f"📦 Context vectors shape: {context_vectors.shape}")
    print("\n💡 Key Observations:")
    print("  • Attention weights sum to 1 for each query")
    print("  • Higher weights indicate more relevant key positions")
    print("  • Context vectors are weighted combinations of values")
    
    return attention_weights, context_vectors

# Run the demonstration
attention_weights, context_vectors = attention_mechanism_demo()

## Chapter Summary

### Key Takeaways

1. **Attention Fundamentals**: Attention mechanisms enable models to dynamically focus on relevant parts of the input sequence, overcoming limitations of fixed-size context vectors in traditional encoder-decoder architectures.

2. **Mathematical Foundation**: The scaled dot-product attention mechanism provides a computationally efficient way to compute compatibility between queries and keys, with the scaling factor preventing gradient vanishing.

3. **Multi-Head Attention**: Multiple attention heads allow the model to jointly attend to information from different representation subspaces, enhancing the modeling capability for complex motor behaviors.

### Next Steps

In the next chapter, we will implement complete attention-based models for motor performance map prediction, integrating all the concepts covered in this chapter with practical considerations for real-world deployment.